# Multidimensional Welfare Index for Russian Regions (2020–2023)

This notebook builds an **Integrated Welfare Index** for 84 Russian federal subjects using data from Rosstat.

## Methodology
1. **Data collection** — 12 indicators across 4 thematic blocks from Rosstat statistical yearbooks
2. **Min-max normalization** — with inversion for negative-direction indicators
3. **Principal Component Analysis (PCA)** — within each block to derive block scores
4. **Weighted aggregation** — weights derived from explained variance ratios
5. **K-means clustering** — k=4 for regional typology
6. **Hypothesis testing** — three research hypotheses on the structure of regional inequality

## Blocks and Indicators
| Block | Indicators |
|---|---|
| Material wellbeing | Per capita income, Real income index, Poverty rate |
| Property endowment | Housing area per capita, Cars per 1000 persons |
| Human capital | Life expectancy, Infant mortality, Preschool coverage, University students |
| Digitalization | Internet access (households), Internet access (population), Internet use (organizations) |

## Data Source
Rosstat. *Regions of Russia. Socio-Economic Indicators* — 2021, 2022, 2023, 2024 editions.  
URL: https://rosstat.gov.ru

---

## 0. Setup

In [ ]:
# Install dependencies if needed
# !pip install pandas scikit-learn matplotlib seaborn openpyxl scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plotting settings
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

# Color palette for 4 clusters
CLUSTER_COLORS = {1: '#2166ac', 2: '#74add1', 3: '#fdae61', 4: '#d73027'}
COLORS_LIST = ['#2166ac', '#74add1', '#fdae61', '#d73027']

print('Libraries loaded successfully')

## 1. Block and Indicator Definitions

In [ ]:
# Block structure: indicator name (as in Excel) -> direction
# +1 = higher is better, -1 = lower is better (will be inverted)

BLOCKS = {
    'Material Wellbeing': {
        'Среднедуш. доходы, руб/мес':       +1,   # Per capita income, RUB/month
        'Реальные доходы, % к пред. году':  +1,   # Real income index, % of prev. year
        'Бедность, % населения':            -1,   # Poverty rate, % of population
    },
    'Property Endowment': {
        'Легковые авто на 1000 чел.':       +1,   # Cars per 1000 persons
        'Площадь жилья, кв.м/чел':          +1,   # Housing area, sq.m per capita
    },
    'Human Capital': {
        'ОПЖ, лет':                         +1,   # Life expectancy at birth, years
        'Младенч. смертность, на 1000':     -1,   # Infant mortality per 1000 live births
        'Охват дошк. образованием, %':      +1,   # Preschool education coverage, %
        'Студенты вузов на 10000 чел.':     +1,   # University students per 10,000 persons
    },
    'Digitalization': {
        'Интернет: население, %':           +1,   # Internet users among population, %
        'Интернет: домохозяйства, %':       +1,   # Households with internet access, %
        'Интернет: организации, %':         +1,   # Organizations using internet, %
    },
}

# Regions excluded from analysis
EXCLUDED_REGIONS = [
    'Ямало-Ненецкий',        # Extreme income values (oil & gas rent)
    'Ненецкий автономный',   # Extreme income values
    'Чукотский',             # Systematic data gaps
    'Донецкая Народная',     # New subjects (incomplete data)
    'Луганская Народная',
    'Запорожская',
    'Херсонская',
    'Архангельская область без',  # Duplicate entry in Rosstat
]

ANALYSIS_YEAR = 2023
ALL_YEARS = [2020, 2021, 2022, 2023]

print(f'Blocks defined: {list(BLOCKS.keys())}')
print(f'Total indicators: {sum(len(v) for v in BLOCKS.values())}')

## 2. Data Loading

In [ ]:
def load_data(filepath: str, year: int) -> pd.DataFrame:
    """
    Load data from the Rosstat Excel file for a given year.
    
    Parameters
    ----------
    filepath : str
        Path to data_rosstat.xlsx
    year : int
        Year to load (2020–2023)
    
    Returns
    -------
    pd.DataFrame
        Clean DataFrame: regions as index, indicators as columns
    """
    df = pd.read_excel(filepath, sheet_name=f'summary_{year}', index_col=0)
    
    # Remove excluded regions
    mask = df.index.str.contains('|'.join(EXCLUDED_REGIONS), na=False)
    df = df[~mask]
    
    # Drop rows with more than 30% missing values
    thresh = int(df.shape[1] * 0.7)
    df = df.dropna(thresh=thresh)
    
    # Fill remaining gaps with column median
    df = df.fillna(df.median())
    
    print(f'Year {year}: {len(df)} regions, {df.shape[1]} indicators, '
          f'{df.isnull().sum().sum()} missing values after cleaning')
    return df


# Load main analysis year
DATA_PATH = 'data_rosstat.xlsx'  # adjust path if needed

df_main = load_data(DATA_PATH, ANALYSIS_YEAR)
df_main.head(3)

## 3. Descriptive Statistics

In [ ]:
# All indicators present in the data
all_indicators = {k: v for block in BLOCKS.values() for k, v in block.items()}
available_cols = [c for c in all_indicators if c in df_main.columns]

desc = df_main[available_cols].describe().T[['min', 'max', 'mean', '50%', 'std']]
desc.columns = ['Min', 'Max', 'Mean', 'Median', 'Std']
desc = desc.round(2)

print('=== Descriptive Statistics (2023, 84 regions) ===')
print(desc.to_string())

## 4. Normalization

In [ ]:
def normalize(df: pd.DataFrame, blocks: dict) -> pd.DataFrame:
    """
    Min-max normalization to [0, 1].
    Indicators with direction=-1 are inverted before normalization
    so that higher normalized value always means better welfare.
    
    Parameters
    ----------
    df : pd.DataFrame
        Raw data
    blocks : dict
        Block structure with directions
    
    Returns
    -------
    pd.DataFrame
        Normalized data in [0, 1]
    """
    df_norm = df.copy()
    for block_indicators in blocks.values():
        for col, direction in block_indicators.items():
            if col not in df_norm.columns:
                continue
            x = df_norm[col]
            if direction == -1:
                x = x.max() + x.min() - x  # inversion
            df_norm[col] = (x - x.min()) / (x.max() - x.min())
    return df_norm


df_norm = normalize(df_main, BLOCKS)
print('Normalization complete. Value range check:')
print(df_norm[available_cols].agg(['min', 'max']).round(4))

## 5. PCA Within Blocks

In [ ]:
def apply_pca_block(
    df_norm: pd.DataFrame,
    block_name: str,
    indicators: dict
) -> tuple:
    """
    Apply PCA to a single block and return the first principal component score.
    
    The sign of PC1 is adjusted so that higher score = better welfare
    (positive average correlation with all normalized indicators).
    
    Returns
    -------
    tuple: (block_score Series, explained_variance_ratio float, loadings dict)
    """
    cols = [c for c in indicators if c in df_norm.columns]
    X = df_norm[cols].values

    if len(cols) == 1:
        return df_norm[cols[0]], 1.0, {cols[0]: 1.0}

    pca = PCA(n_components=1)
    scores = pca.fit_transform(X).flatten()

    # Check sign: PC1 should correlate positively with all indicators
    avg_corr = np.mean([np.corrcoef(scores, X[:, i])[0, 1] for i in range(X.shape[1])])
    scores_norm = (scores - scores.min()) / (scores.max() - scores.min())
    if avg_corr < 0:
        scores_norm = 1 - scores_norm

    ev = pca.explained_variance_ratio_[0]
    loadings = {col: round(abs(pca.components_[0][i]), 3) for i, col in enumerate(cols)}

    return pd.Series(scores_norm, index=df_norm.index), ev, loadings


# Apply PCA to all blocks
block_scores = {}
block_ev = {}
block_loadings = {}

print('=== PCA Results by Block ===')
for block_name, indicators in BLOCKS.items():
    score, ev, loadings = apply_pca_block(df_norm, block_name, indicators)
    block_scores[block_name] = score
    block_ev[block_name] = ev
    block_loadings[block_name] = loadings
    print(f'\n{block_name}:')
    print(f'  Explained variance (PC1): {ev:.3f}')
    for ind, load in loadings.items():
        print(f'  Loading [{ind}]: {load:.3f}')

scores_df = pd.DataFrame(block_scores)

## 6. Index Aggregation

In [ ]:
# Block weights = explained variance ratio (data-driven, no subjective judgment)
total_ev = sum(block_ev.values())
weights = {b: ev / total_ev for b, ev in block_ev.items()}

print('=== Block Weights ===')
for b, w in weights.items():
    print(f'  {b}: {w:.3f}')

# Weighted sum
index = sum(scores_df[b] * w for b, w in weights.items())

# Rescale to [0, 100] for interpretability
index = (index - index.min()) / (index.max() - index.min()) * 100
index.name = 'Welfare Index'

print(f'\n=== Index Summary ({ANALYSIS_YEAR}) ===')
print(f'  Regions: {len(index)}')
print(f'  Median: {index.median():.1f}')
print(f'  Std dev: {index.std():.1f}')
print(f'  Max: {index.max():.1f} ({index.idxmax()})')
print(f'  Min: {index.min():.1f} ({index.idxmin()})')

## 7. Sensitivity Analysis

In [ ]:
# Recalculate with equal weights to test robustness
index_equal = scores_df.mean(axis=1)
index_equal = (index_equal - index_equal.min()) / (index_equal.max() - index_equal.min()) * 100

corr_sensitivity, p_sensitivity = stats.pearsonr(index, index_equal)
print(f'Sensitivity analysis:')
print(f'  Correlation (weighted vs equal weights): r = {corr_sensitivity:.3f}, p = {p_sensitivity:.4f}')
print(f'  -> Index ranking is robust to weighting scheme')

## 8. Clustering (k=4)

In [ ]:
# Silhouette scores for k=2..6
print('=== Silhouette Scores by k ===')
sil_scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(scores_df.values)
    sil_scores[k] = silhouette_score(scores_df.values, labels)
    print(f'  k={k}: silhouette = {sil_scores[k]:.3f}')

print(f'\nOptimal k (silhouette): {max(sil_scores, key=sil_scores.get)}')
print('Using k=4 for substantive interpretability (see paper for rationale)')

# Apply k=4
km4 = KMeans(n_clusters=4, random_state=42, n_init=10)
raw_labels = km4.fit_predict(scores_df.values)

# Rank clusters by descending mean index (Cluster 1 = highest welfare)
cluster_means = {c: index[raw_labels == c].mean() for c in range(4)}
rank_map = {c: r for r, (c, _) in
            enumerate(sorted(cluster_means.items(), key=lambda x: -x[1]), 1)}
clusters = pd.Series([rank_map[l] for l in raw_labels],
                     index=scores_df.index, name='Cluster')

print('\n=== Cluster Profiles ===')
for cl in sorted(clusters.unique()):
    mask = clusters == cl
    vals = index[mask]
    regs = clusters[mask].index.tolist()
    print(f'\nCluster {cl} ({len(regs)} regions): '
          f'index {vals.mean():.1f} ± {vals.std():.1f} [{vals.min():.1f}–{vals.max():.1f}]')
    print(f'  Examples: {", ".join(regs[:5])}')

## 9. Hypothesis Testing

In [ ]:
print('=== Hypothesis Testing ===')

# H1: Regions differ not only in income level but in welfare STRUCTURE
# Evidence: cluster 3 has low material block but relatively high human capital
print('\nH1: Structural heterogeneity')
cluster_profiles = scores_df.copy()
cluster_profiles['Cluster'] = clusters
profile_means = cluster_profiles.groupby('Cluster').mean()
print(profile_means.round(3).to_string())

# H2: Stable geographical and typological patterns
# Evidence: cluster composition matches known regional typologies
print('\nH2: Stable typological patterns — see cluster profiles above')

# H3: Human capital and digitalization contribute independently of income
income_col = 'Среднедуш. доходы, руб/мес'
if income_col in df_main.columns:
    corr_h3, pval_h3 = stats.pearsonr(index, df_main[income_col])
    print(f'\nH3: Correlation of welfare index with income:')
    print(f'  r = {corr_h3:.3f}, p = {pval_h3:.4f}')
    print(f'  -> Moderate correlation confirms index captures variance beyond income')

## 10. Dynamic Analysis (2020–2023)

In [ ]:
def build_index_for_year(filepath: str, year: int) -> pd.Series:
    """Build welfare index for a single year."""
    df = load_data(filepath, year)
    df_n = normalize(df, BLOCKS)
    b_scores, b_ev = {}, {}
    for bname, inds in BLOCKS.items():
        score, ev, _ = apply_pca_block(df_n, bname, inds)
        b_scores[bname] = score
        b_ev[bname] = ev
    tot = sum(b_ev.values())
    w = {b: ev / tot for b, ev in b_ev.items()}
    sc = pd.DataFrame(b_scores)
    idx = sum(sc[b] * wt for b, wt in w.items())
    return (idx - idx.min()) / (idx.max() - idx.min()) * 100


# Build index for all years
yearly_cluster_means = {}
for yr in ALL_YEARS:
    idx_yr = build_index_for_year(DATA_PATH, yr)
    common = clusters.index.intersection(idx_yr.index)
    for cl in sorted(clusters.unique()):
        mask = clusters[common] == cl
        yearly_cluster_means.setdefault(cl, {})[yr] = idx_yr[common][mask].mean()

print('Dynamic analysis complete.')
for cl in sorted(clusters.unique()):
    vals = [f"{yr}: {yearly_cluster_means[cl][yr]:.1f}" for yr in ALL_YEARS]
    print(f'  Cluster {cl}: {", ".join(vals)}')

## 11. Visualizations

### Fig 1 — Correlation Heatmap

In [ ]:
short_names = {
    'Среднедуш. доходы, руб/мес':       'Income',
    'Реальные доходы, % к пред. году':  'Real income',
    'Бедность, % населения':            'Poverty',
    'Легковые авто на 1000 чел.':       'Cars',
    'Площадь жилья, кв.м/чел':          'Housing',
    'ОПЖ, лет':                         'Life exp.',
    'Младенч. смертность, на 1000':     'Infant mort.',
    'Охват дошк. образованием, %':      'Preschool',
    'Студенты вузов на 10000 чел.':     'Students',
    'Интернет: население, %':           'Internet (pop.)',
    'Интернет: домохозяйства, %':       'Internet (HH)',
    'Интернет: организации, %':         'Internet (org.)',
}

corr_matrix = df_main[available_cols].rename(columns=short_names).corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlBu_r', center=0, vmin=-1, vmax=1,
            ax=ax, linewidths=0.5,
            annot_kws={'size': 8}, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation matrix of welfare indicators (2023)', fontsize=12, pad=12)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=8)
plt.tight_layout()
plt.savefig('fig1_corr_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### Fig 2 — Top-15 and Bottom-15 Regions

In [ ]:
top = index.nlargest(15).sort_values()
bottom = index.nsmallest(15).sort_values()
combined = pd.concat([bottom, top])

fig, ax = plt.subplots(figsize=(11, 10))
bar_colors = [CLUSTER_COLORS[clusters[r]] for r in combined.index]
vals_display = np.where(combined.values < 0.8, 0.8, combined.values)

ax.barh(range(len(combined)), vals_display,
        color=bar_colors, edgecolor='white', height=0.75)

# Value labels
for i, (val, v) in enumerate(zip(combined.values, vals_display)):
    ax.text(v + 0.5, i, f'{val:.1f}', va='center', fontsize=7.5, color='#333333')

ax.set_yticks(range(len(combined)))
ax.set_yticklabels([r[:40] for r in combined.index], fontsize=8.5)
ax.axvline(index.median(), color='gray', linestyle='--',
           alpha=0.7, linewidth=1.5, label=f'Median ({index.median():.1f})')

patches = [mpatches.Patch(color=CLUSTER_COLORS[i], label=f'Cluster {i}') for i in range(1, 5)]
ax.legend(handles=patches + [ax.get_lines()[0]], fontsize=8, loc='lower right')
ax.set_xlabel('Welfare Index (0–100)', fontsize=10)
ax.set_title('Top-15 and bottom-15 regions by welfare index (2023)', fontsize=11, pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig2_top_bottom.png', dpi=150, bbox_inches='tight')
plt.show()

### Fig 3 — Welfare Index by Federal District (Boxplot)

In [ ]:
DISTRICT_MAP = {
    'Central':         ['Белгородская','Брянская','Владимирская','Воронежская','Ивановская',
                        'Калужская','Костромская','Курская','Липецкая','Московская','Орловская',
                        'Рязанская','Смоленская','Тамбовская','Тверская','Тульская','Ярославская','Москва'],
    'Northwest':       ['Карелия','Коми','Архангельская','Вологодская','Калининградская',
                        'Ленинградская','Мурманская','Новгородская','Псковская','Санкт-Петербург'],
    'South':           ['Адыгея','Калмыкия','Краснодарский','Астраханская','Волгоградская',
                        'Ростовская','Крым','Севастополь'],
    'North Caucasus':  ['Дагестан','Ингушетия','Кабардино','Карачаево','Северная Осетия',
                        'Чеченская','Ставропольский'],
    'Volga':           ['Башкортостан','Марий Эл','Мордовия','Татарстан','Удмуртская',
                        'Чувашская','Пермский','Кировская','Нижегородская','Оренбургская',
                        'Пензенская','Самарская','Саратовская','Ульяновская'],
    'Ural':            ['Курганская','Свердловская','Тюменская','Челябинская','Ханты-Мансийский'],
    'Siberian':        ['Алтай','Тыва','Хакасия','Алтайский','Красноярский','Иркутская',
                        'Кемеровская','Новосибирская','Омская','Томская'],
    'Far East':        ['Бурятия','Якутия','Забайкальский','Камчатский','Приморский',
                        'Хабаровский','Амурская','Магаданская','Сахалинская','Еврейская'],
}

def get_district(region):
    for dist, keywords in DISTRICT_MAP.items():
        if any(kw.lower() in region.lower() for kw in keywords):
            return dist
    return 'Other'

idx_df = index.reset_index()
idx_df.columns = ['region', 'index']
idx_df['district'] = idx_df['region'].apply(get_district)
idx_df = idx_df[idx_df['district'] != 'Other']

order = idx_df.groupby('district')['index'].median().sort_values(ascending=False).index.tolist()

fig, ax = plt.subplots(figsize=(11, 6))
ax.boxplot([idx_df[idx_df['district'] == d]['index'].values for d in order],
           labels=order, patch_artist=True,
           boxprops=dict(facecolor='#d1e5f0', color='#2166ac'),
           medianprops=dict(color='#d73027', linewidth=2),
           whiskerprops=dict(color='#2166ac'),
           capprops=dict(color='#2166ac'),
           flierprops=dict(marker='o', color='#2166ac', alpha=0.5, markersize=5))
ax.set_ylabel('Welfare Index (0–100)', fontsize=10)
ax.set_xlabel('Federal District', fontsize=10)
ax.set_title('Distribution of welfare index by federal district (2023)', fontsize=11, pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('fig3_district_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

### Fig 4 — Sensitivity Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for cl in sorted(clusters.unique()):
    mask = clusters == cl
    ax.scatter(index[mask], index_equal[mask],
               color=COLORS_LIST[cl - 1], alpha=0.65, s=45, label=f'Cluster {cl}')
ax.plot([0, 105], [0, 105], 'k--', alpha=0.4, linewidth=1.5, label='y=x')
ax.set_xlabel('Index (variance-weighted)', fontsize=10)
ax.set_ylabel('Index (equal weights)', fontsize=10)
ax.set_title(f'Sensitivity analysis (r={corr_sensitivity:.3f})', fontsize=11)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig4_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

### Fig 5 — Welfare Index vs Income (H3)

In [ ]:
income = df_main[income_col]
z = np.polyfit(income, index, 1)
p_line = np.poly1d(z)
x_line = np.linspace(income.min(), income.max(), 100)

fig, ax = plt.subplots(figsize=(9, 6))
for cl in sorted(clusters.unique()):
    mask = clusters == cl
    ax.scatter(income[mask], index[mask],
               color=COLORS_LIST[cl - 1], alpha=0.7, s=55,
               label=f'Cluster {cl}', edgecolors='white', linewidth=0.5)
ax.plot(x_line, p_line(x_line), 'k--', alpha=0.4, linewidth=1.5, label='Trend line')

# Label notable regions
labels = {
    'г. Москва': (2000, -3),
    'Камчатский край': (1500, 2),
    'Республика Ингушетия': (1500, 2),
    'Кировская область': (1000, 2),
}
for reg, (dx, dy) in labels.items():
    matches = [r for r in index.index if reg.lower() in r.lower()]
    if matches:
        r = matches[0]
        ax.annotate(r.replace('Республика ', 'R. ').replace('г. ', '')[:18],
                    xy=(income[r], index[r]),
                    xytext=(income[r] + dx, index[r] + dy),
                    fontsize=7.5, color='#333333',
                    arrowprops=dict(arrowstyle='->', color='#aaaaaa', lw=0.8))

ax.set_xlabel('Per capita income, RUB/month (2023)', fontsize=10)
ax.set_ylabel('Welfare Index (0–100)', fontsize=10)
ax.set_title(f'Welfare index vs income (r = {corr_h3:.3f}, p < 0.001)', fontsize=11, pad=10)
ax.legend(fontsize=9, loc='upper left')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(alpha=0.2, linestyle='--')
plt.tight_layout()
plt.savefig('fig5_income_vs_index.png', dpi=150, bbox_inches='tight')
plt.show()

### Fig 6 — Cluster Profiles by Block

In [ ]:
BLOCK_SHORT = {
    'Material Wellbeing':   'Material\nWellbeing',
    'Property Endowment':   'Property\nEndowment',
    'Human Capital':        'Human\nCapital',
    'Digitalization':       'Digitalization',
}

cp = scores_df.copy()
cp['Cluster'] = clusters
profiles = cp.groupby('Cluster').mean()

x = np.arange(len(scores_df.columns))
width = 0.18

fig, ax = plt.subplots(figsize=(11, 6))
for i, (cl, row) in enumerate(profiles.iterrows()):
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, row.values, width,
                  label=f'Cluster {cl}', color=COLORS_LIST[i],
                  alpha=0.88, edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                f'{h:.2f}', ha='center', va='bottom', fontsize=7, color='#444')

ax.set_xticks(x)
ax.set_xticklabels([BLOCK_SHORT.get(b, b) for b in scores_df.columns],
                   fontsize=10, ha='center')
ax.set_ylabel('Average block score (0–1)', fontsize=10)
ax.set_title('Cluster profiles by welfare block (2023)', fontsize=11, pad=12)
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(0, 1.12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.25, linestyle='--')
plt.tight_layout()
plt.savefig('fig6_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

### Fig 7 — Material Wellbeing vs Human Capital (H1)

In [ ]:
b_mat = 'Material Wellbeing'
b_hum = 'Human Capital'

fig, ax = plt.subplots(figsize=(8, 7))
for cl in sorted(clusters.unique()):
    mask = clusters == cl
    ax.scatter(scores_df.loc[mask, b_mat], scores_df.loc[mask, b_hum],
               color=COLORS_LIST[cl - 1], alpha=0.7, s=55,
               label=f'Cluster {cl}', edgecolors='white', linewidth=0.5)

label_regions = {
    'Республика Ингушетия': (-0.03, 0.04),
    'г. Москва': (-0.03, -0.05),
    'Камчатский край': (0.02, 0.01),
    'Республика Тыва': (0.02, 0.01),
}
for reg, (dx, dy) in label_regions.items():
    matches = [r for r in scores_df.index if reg.lower() in r.lower()]
    if matches:
        r = matches[0]
        ax.annotate(r.replace('Республика ', 'R. ')[:18],
                    xy=(scores_df.loc[r, b_mat], scores_df.loc[r, b_hum]),
                    xytext=(scores_df.loc[r, b_mat] + dx, scores_df.loc[r, b_hum] + dy),
                    fontsize=7, arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

ax.set_xlabel('Block: Material Wellbeing', fontsize=10)
ax.set_ylabel('Block: Human Capital', fontsize=10)
ax.set_title('Material wellbeing vs human capital by cluster (2023)', fontsize=11, pad=10)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig7_scatter_mat_hum.png', dpi=150, bbox_inches='tight')
plt.show()

### Fig 8 — Dynamic Analysis 2020–2023

In [ ]:
CL_LABELS = {
    1: 'Cluster 1 (high)',
    2: 'Cluster 2 (above average)',
    3: 'Cluster 3 (specific profile)',
    4: 'Cluster 4 (below average)',
}
MARKERS = ['o', 's', '^', 'D']

fig, ax = plt.subplots(figsize=(10, 5.5))
for cl in sorted(clusters.unique()):
    yvals = [yearly_cluster_means[cl][yr] for yr in ALL_YEARS]
    ax.plot(ALL_YEARS, yvals,
            color=COLORS_LIST[cl - 1], marker=MARKERS[cl - 1],
            linewidth=2.2, markersize=7, label=CL_LABELS[cl])
    ax.annotate(f'{yvals[-1]:.1f}',
                xy=(2023, yvals[-1]),
                xytext=(2023.08, yvals[-1]),
                fontsize=8.5, color=COLORS_LIST[cl - 1], va='center')

ax.set_xticks(ALL_YEARS)
ax.set_xlim(2019.7, 2024.0)
ax.set_xlabel('Year', fontsize=10)
ax.set_ylabel('Mean welfare index (0–100)', fontsize=10)
ax.set_title('Dynamics of welfare index by cluster (2020–2023)', fontsize=11, pad=10)
ax.legend(fontsize=9, loc='upper center',
          bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('fig8_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Export Results

In [ ]:
# Final ranking table
result_df = pd.DataFrame({
    'Welfare Index': index.round(2),
    'Rank': index.rank(ascending=False).astype(int),
    'Cluster': clusters,
})
for b in scores_df.columns:
    result_df[f'Block: {b}'] = scores_df[b].round(3)
result_df = result_df.sort_values('Rank')

# PCA loadings table
loads_rows = []
for block, loads in block_loadings.items():
    for ind, val in loads.items():
        loads_rows.append({
            'Block': block,
            'Indicator': ind,
            'PC1 Loading': val,
            'Explained Variance': round(block_ev[block], 3),
            'Block Weight': round(weights[block], 3),
        })
loads_df = pd.DataFrame(loads_rows)

# Cluster statistics
cl_df = scores_df.copy()
cl_df['Index'] = index
cl_df['Cluster'] = clusters
cl_stats = cl_df.groupby('Cluster')['Index'].agg(['mean', 'std', 'min', 'max', 'count'])

# Save
with pd.ExcelWriter('index_results.xlsx', engine='openpyxl') as writer:
    result_df.to_excel(writer, sheet_name='Regional Ranking')
    loads_df.to_excel(writer, sheet_name='PCA Loadings', index=False)
    cl_stats.to_excel(writer, sheet_name='Cluster Stats')
    scores_df.assign(Index=index, Cluster=clusters).sort_values(
        'Index', ascending=False
    ).to_excel(writer, sheet_name='Block Scores')
    df_main.describe().round(2).to_excel(writer, sheet_name='Descriptive Stats')

print('Results saved to index_results.xlsx')
print('\nTop-10 regions:')
print(result_df[['Welfare Index', 'Rank', 'Cluster']].head(10).to_string())

---
## Summary

### Key Findings

| Metric | Value |
|---|---|
| Regions analyzed | 84 |
| Time period | 2020–2023 |
| Median index (2023) | ~46 |
| Max–min range | 100 points |
| Sensitivity (r) | 0.977 |
| Correlation with income | 0.513 |

### Hypotheses
- **H1** ✅ Regions differ in welfare *structure*, not only in income level
- **H2** ✅ Stable geographical and typological patterns confirmed
- **H3** ✅ Digitalization and human capital contribute independently of income (r=0.513)

### References
- Aivazyan S.A., Afanasyev M.Yu., Kudrov A.V. (2023). Integral indicator of quality of living conditions in Russian regions. *Economy of Region*, 19(1), 17–32.
- Zubarevich N.V. (2019). *Socio-economic development of Russian regions: inequality and growth factors*. Moscow: NISP.
- Sen A. (1999). *Development as Freedom*. New York: Oxford University Press.
- OECD (2008). *Handbook on Constructing Composite Indicators*. Paris: OECD Publishing.
- UNDP (2024). *Human Development Report 2023/2024*. New York: UNDP.
- Rosstat (2024). *Regions of Russia. Socio-Economic Indicators — 2024*. Moscow: Rosstat.